In [ ]:
## Model Recovery and Confusion Matrix

The confusion matrix tests whether AIC and BIC can actually distinguish these specific models with this amount of data. 

A strong diagonal means our pipeline correctly distinguishes the models.

In [ ]:
def model_recovery(n_sims=50):
    sim_funcs = [
        lambda: simulate_model1(0.3, -8),
        lambda: simulate_model2(0.4, -5, 0.5),
        lambda: simulate_model3(0.25, 0.35, -6),
    ]
    nll_funcs = [nll_model1, nll_model2, nll_model3]
    x0s       = [[0.3, -8], [0.4, -5, 0.5], [0.25, 0.35, -6]]
    n_par     = [2, 3, 3]
    names     = ["M1", "M2", "M3"]

    conf = np.zeros((3, 3), dtype=int)
    print(f"Running model recovery ({n_sims} sims per model)...")

    for gi in range(3):
        for _ in range(n_sims):
            # Simulate one dataset from the generating model
            sc, so = sim_funcs[gi]()

            # Fit all three models to this simulated dataset
            best_bic, best = np.inf, 0
            for fi in range(3):
                res = fit_model(nll_funcs[fi], sc, so, x0s[fi])
                # Select the winner by BIC (lower = better)
                _, bic = aic_bic(res.fun, n_par[fi])
                if bic < best_bic: best_bic, best = bic, fi

            # Record which model won
            conf[gi, best] += 1
        print(f"  {names[gi]} done")

    # Display confusion matrix
    print(f"\nConfusion matrix (rows=true, cols=recovered):")
    print(pd.DataFrame(conf, index=names, columns=names))

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(conf, annot=True, fmt="d", cmap="Blues",
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Recovered"); ax.set_ylabel("True")
    ax.set_title("Model Recovery")
    plt.tight_layout(); plt.savefig("figures/task_j.png"); plt.show()
    return conf

confusion = model_recovery(50)

# When data is generated by Model 1, BIC correctly selects it 47 out of 50 times. However, when data is generated by Model 2, BIC selects Model 1 in 48 out of 50 cases and never selects Model 2. Similarly, when data is generated by Model 3, BIC selects Model 1 in 45 out of 50 cases and only correctly identifies Model 3 three times. The extra parameters in Models 2 and 3 do not produce sufficiently distinct behaviour (in 160 trials) for BIC to justify the added complexity.

In [ ]:
## Discussion

### Models

- Model 1 tests whether anxiety alters overall learning speed (α) or decision noise (β).
- Model 2 tests whether anxious individuals forget learned associations faster (low A).
- Model 3 is often the most informative: elevated α⁻ in anxious participants implies hypersensitivity to punishment, while reduced α⁺ implies impaired safety learning.

### Clinical Implications

If a specific learning rate parameter differs between groups, this suggests targeted interventions. For example, if the core deficit is low α⁺ (poor safety learning), exposure therapy could be designed to maximise non-punished trials.